# Entrenamiento YOLO: detección de plagas del tomate (Majes)

Compara **YOLOv8n, YOLOv10n y YOLO11n** con las mismas opciones sobre una versión de Roboflow, con la
división por planta intacta.

**Antes de empezar**

1. Menú *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*.
2. En Roboflow genera la versión: sin Rebalance, Auto-Orient, Resize **Fit within 1024×1024**
   (nunca Stretch) y **sin** aumentos.
3. Revisa la celda de **configuración** y luego usa *Ejecutar todas*. Te pedirá permiso de Drive y tu API
   key de Roboflow (la escribes tú; no queda guardada en el notebook).

**Si Colab se desconecta:** vuelve a *Ejecutar todas*. Todo se guarda en Drive: las corridas terminadas
se saltan y la que quedó a medias continúa desde su última época.

In [ ]:
!nvidia-smi -L
!pip install -q ultralytics==8.4.82 roboflow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configuración

In [ ]:
from pathlib import Path

NOMBRE = 'v5-1966-completas'   # nombre de esta tanda; cámbialo en cada versión del dataset
VERSION = 2                    # número de versión en Roboflow
ESPERADO = {'train': 1370, 'valid': 404, 'test': 192}   # imágenes por subconjunto de esta versión

MODELOS = ['yolov8n.pt', 'yolov10n.pt', 'yolo11n.pt']
SEMILLAS = [0, 1, 2]           # primero se completa la semilla 0 de los 3 modelos
EPOCHS = 150
PATIENCE = 30
IMGSZ = 640                    # la app usa 640: no cambiar sin ajustar la app
BATCH = 16
DEVICE = 0

CLASES = ['mildiu', 'mosca_blanca', 'oidio', 'polilla', 'sana']   # mismo orden que la app y el backend

DRIVE = Path('/content/drive/MyDrive/tomate_entrenamiento')
RES = DRIVE / NOMBRE
DATASET = Path('/content/dataset')
TRABAJO = Path('/content/trabajo')

## Descargar el dataset

In [ ]:
from getpass import getpass
from roboflow import Roboflow

if not (DATASET / 'data.yaml').exists():
    rf = Roboflow(api_key=getpass('API key de Roboflow: '))
    rf.workspace('fray-hxtls').project('plagas-tomate-majes-v2').version(VERSION).download(
        'yolov8', location=str(DATASET), overwrite=True)

## Verificar el dataset

Se detiene con un error si algo invalidaría los resultados: clases en otro orden, conteos distintos,
imágenes deformadas o una misma planta en dos subconjuntos.

In [ ]:
import shutil
import collections
import re

import yaml
from PIL import Image

EXTS = {'.jpg', '.jpeg', '.png'}


def nombre_base(p):
    # Roboflow agrega _jpg.rf.<hash> al nombre original
    return re.sub(r'_(jpg|jpeg|png)\.rf\..*$', '', Path(p).stem, flags=re.I)


def grupo(p):
    # <carpeta>__g015__20260503_120027 -> <carpeta>__g015 (planta o video)
    return nombre_base(p).rsplit('__', 1)[0]


def periodo(p):
    # Hora real de captura, leida del nombre: AAAAMMDD_HHMMSS
    hora = int(re.findall(r'\d{8}_(\d{2})\d{4}', nombre_base(p))[-1])
    return 'manana' if hora < 11 else 'mediodia' if hora < 14 else 'tarde'


def imagenes(split):
    return sorted(p for p in (DATASET / split / 'images').iterdir() if p.suffix.lower() in EXTS)


def etiqueta(p):
    return p.parent.parent / 'labels' / (p.stem + '.txt')


def clases_en(p):
    t = etiqueta(p)
    return [int(l.split()[0]) for l in t.read_text().splitlines() if l.strip()] if t.exists() else []


info = yaml.safe_load((DATASET / 'data.yaml').read_text())
assert list(info['names']) == CLASES, f"Orden de clases distinto al de la app: {info['names']}"

# Correccion de fuga video <-> fotos (bitacora 6b): 66 imagenes que van a otro subconjunto.
# En Roboflow tienen el tag mover-a-train / mover-a-valid / mover-a-test. Si ya se movieron alla, esto no hace nada.
MOVER = {
    'frames_extraidosv1__video_20260503_143122__20260503_143122_frame_002430': 'valid',
    'frames_extraidosv1__video_20260503_174007__20260503_174007_frame_000330': 'train',
    'frames_extraidosv1__video_20260503_174007__20260503_174007_frame_001770': 'train',
    'frames_extraidosv1__video_20260503_173624__20260503_173624_frame_000360': 'train',
    'frames_extraidosv1__video_20260503_173624__20260503_173624_frame_002940': 'train',
    'frames_extraidosv1__video_20260503_173645__20260503_173645_frame_002940': 'train',
    '16__g041__20260503_171906': 'train',
    '16__g041__20260503_171924': 'train',
    '16__g041__20260503_171943': 'train',
    '16__g041__20260503_172003': 'train',
    '16__g041__20260503_172016': 'train',
    '16__g041__20260503_172025': 'train',
    'frames_extraidosv1__video_20260503_171654__20260503_171654_frame_000000': 'train',
    '16__g029__20260503_165433': 'train',
    '16__g029__20260503_165439': 'train',
    '16__g031__20260503_165815': 'train',
    '16__g031__20260503_165821': 'train',
    '16__g031__20260503_165848': 'train',
    '16__g031__20260503_165858': 'train',
    '16__g031__20260503_165903': 'train',
    '16__g031__20260503_165907': 'train',
    'frames_extraidosv1__video_20260503_165721__20260503_165721_frame_000000': 'train',
    'frames_extraidosv1__video_20260503_165721__20260503_165721_frame_000060': 'train',
    'frames_extraidosv1__video_20260503_165721__20260503_165721_frame_000660': 'train',
    'frames_extraidosv1__video_20260503_165721__20260503_165721_frame_001230': 'train',
    '16__g008__20260503_161618': 'train',
    '16__g008__20260503_161621': 'train',
    '16__g025__20260503_164855': 'train',
    '16__g025__20260503_164858': 'train',
    '16__g025__20260503_164906': 'train',
    '16__g025__20260503_164915': 'train',
    '16__g026__20260503_165015': 'train',
    '16__g026__20260503_165030': 'train',
    'frames_extraidosv1__video_20260503_164830__20260503_164830_frame_000000': 'train',
    'frames_extraidosv1__video_20260503_162752__20260503_162752_frame_007290': 'valid',
    'frames_extraidosv1__video_20260503_100125__20260503_100125_frame_000000': 'train',
    'frames_extraidosv1__video_20260503_100125__20260503_100125_frame_000600': 'train',
    'frames_extraidosv1__video_20260503_100125__20260503_100125_frame_000810': 'train',
    '8__g045__20260503_091145': 'valid',
    'version2__V3__4__g016__20260729_151950': 'train',
    'version2__V3__frames_videosv2__video_20260729_151912__20260729_151912_frame_000390': 'train',
    'version2__V3__frames_videosv2__video_20260729_153942__20260729_153942_frame_000210': 'train',
    'version2__V3__frames_videosv2__video_20260729_153942__20260729_153942_frame_000570': 'train',
    'version2__V3__frames_videosv2__video_20260729_144721__20260729_144721_frame_000000': 'valid',
    'version2__V3__frames_videosv2__video_20260729_144721__20260729_144721_frame_000030': 'valid',
    'version2__V3__frames_videosv2__video_20260729_144721__20260729_144721_frame_000060': 'valid',
    'version2__V3__frames_videosv2__video_20260729_144721__20260729_144721_frame_000090': 'valid',
    'version2__V3__frames_videosv2__video_20260729_144721__20260729_144721_frame_000150': 'valid',
    'version2__V3__frames_videosv2__video_20260729_150335__20260729_150335_frame_000060': 'train',
    'version2__V3__frames_videosv2__video_20260729_150335__20260729_150335_frame_000240': 'train',
    'version2__V3__frames_videosv2__video_20260729_150335__20260729_150335_frame_000600': 'train',
    'version2__V3__frames_videosv2__video_20260729_150335__20260729_150335_frame_004200': 'train',
    'version2__V3__frames_videosv2__video_20260729_150335__20260729_150335_frame_004980': 'train',
    'version2__V3__frames_videosv2__video_20260729_150335__20260729_150335_frame_005430': 'train',
    'version2__V3__frames_videosv2__video_20260729_150335__20260729_150335_frame_005550': 'train',
    'version2__V3__4__g014__20260729_151159': 'valid',
    'version2__V3__4__g014__20260729_151205': 'valid',
    'version2__V3__4__g014__20260729_151206': 'valid',
    'version2__V3__4__g014__20260729_151207': 'valid',
    'version2__V3__4__g014__20260729_151208': 'valid',
    'version2__V3__4__g014__20260729_151209': 'valid',
    'version2__V3__4__g004__20260729_143539': 'test',
    'version2__V3__4__g004__20260729_143543': 'test',
    'version2__V3__4__g004__20260729_143547': 'test',
    'version2__V3__4__g010__20260729_145436': 'train',
    'version2__V3__frames_videosv2__video_20260729_151053__20260729_151053_frame_000060': 'test',
}

conteo_roboflow = {s: len(imagenes(s)) for s in ('train', 'valid', 'test')}
print('Imagenes (como vienen de Roboflow):', conteo_roboflow)
assert conteo_roboflow == ESPERADO, f'Se esperaba {ESPERADO}. ¿Se uso Rebalance o es otra version?'
movidas = 0
for s in ('train', 'valid', 'test'):
    for p in imagenes(s):
        destino = MOVER.get(nombre_base(p), s)
        if destino != s:
            for origen, carpeta in ((p, 'images'), (etiqueta(p), 'labels')):
                if origen.exists():
                    (DATASET / destino / carpeta).mkdir(parents=True, exist_ok=True)
                    shutil.move(str(origen), str(DATASET / destino / carpeta / origen.name))
            movidas += 1
print(f'Imagenes movidas de subconjunto por fuga video-fotos: {movidas}')

SPLITS = {s: imagenes(s) for s in ('train', 'valid', 'test')}
conteo = {s: len(v) for s, v in SPLITS.items()}
print('Imagenes:', conteo)

donde = collections.defaultdict(set)
for s, v in SPLITS.items():
    for p in v:
        donde[grupo(p)].add(s)
fuga = {g: sorted(s) for g, s in donde.items() if len(s) > 1}
assert not fuga, f'Fuga de datos: {len(fuga)} grupos en mas de un subconjunto, ej. {list(fuga.items())[:3]}'
print(f'Grupos (plantas/videos): {len(donde)}, ninguno repartido entre subconjuntos')

# Fuga video <-> rafaga de fotos: un video grabado a < 30 s de una rafaga es casi seguro la misma planta
import datetime as dt


def rango_grupo(rutas):
    nb_ = [nombre_base(p) for p in rutas]
    if '__video_' in nb_[0]:
        ini = dt.datetime.strptime(re.search(r'video_(\d{8}_\d{6})', nb_[0]).group(1), '%Y%m%d_%H%M%S')
        cu = [int(re.search(r'frame_(\d+)', n).group(1)) for n in nb_]
        return True, ini + dt.timedelta(seconds=min(cu) / 30), ini + dt.timedelta(seconds=max(cu) / 30)
    t = sorted(dt.datetime.strptime(re.findall(r'\d{8}_\d{6}', n)[-1], '%Y%m%d_%H%M%S') for n in nb_)
    return False, t[0], t[-1]


por_grupo = collections.defaultdict(list)
for s, v in SPLITS.items():
    for p in v:
        por_grupo[grupo(p)].append(p)
rg = {g: (*rango_grupo(v), next(iter(donde[g]))) for g, v in por_grupo.items()}
gap = dt.timedelta(seconds=30)
cerca = [(g, h) for g, (vid, a0, a1, s) in rg.items() if vid
         for h, (vid2, b0, b1, t) in rg.items() if not vid2 and t != s and b0 - gap <= a1 and b1 + gap >= a0]
assert not cerca, f'Fuga video-fotos: {len(cerca)} pares en distinto subconjunto, ej. {cerca[:3]}. Aplicar movimientos_split_v5.csv en Roboflow.'
print('Videos y rafagas cercanas: todos en el mismo subconjunto')

muestra = SPLITS['train'][:: max(1, len(SPLITS['train']) // 50)]
cuadradas = sum(Image.open(p).width == Image.open(p).height for p in muestra)
assert cuadradas < len(muestra) / 2, 'Imagenes cuadradas: la version parece hecha con Stretch. Generala con Fit within.'

print('\nCajas por clase:')
for s, v in SPLITS.items():
    c = collections.Counter(k for p in v for k in clases_en(p))
    print(f'  {s:6}', {CLASES[k]: c[k] for k in range(len(CLASES))})
print('\nImagenes de test por franja:', dict(collections.Counter(periodo(p) for p in SPLITS['test'])))

TRABAJO.mkdir(parents=True, exist_ok=True)
RES.mkdir(parents=True, exist_ok=True)


def escribir_yaml(nombre, test='test/images'):
    # Rutas absolutas: las de Roboflow son relativas y fallan segun la carpeta actual
    f = TRABAJO / f'{nombre}.yaml'
    f.write_text(yaml.safe_dump({
        'path': str(DATASET), 'train': 'train/images', 'val': 'valid/images', 'test': test,
        'names': dict(enumerate(CLASES)),
    }))
    return f


def subconjunto(nombre, rutas):
    # test = lista de imagenes: asi se evalua cada franja por separado
    lista = TRABAJO / f'{nombre}.txt'
    lista.write_text('\n'.join(str(p) for p in rutas) + '\n')
    return escribir_yaml(nombre, test=str(lista))


DATA = escribir_yaml('data')

# Test fijo = las 102 imagenes de test de la v4 (tareas/test_fijo_v4.txt): todas las versiones se
# comparan con estas mismas imagenes. Las 5 que salen de test por la fuga video-fotos quedan fuera solas.
fijos = set([
    '12__g008__20260503_121120',
    '12__g008__20260503_121130',
    '12__g008__20260503_121158',
    '12__g008__20260503_121218',
    '12__g008__20260503_121243',
    '12__g010__20260503_121522',
    '12__g010__20260503_121526',
    '12__g010__20260503_121543',
    '12__g012__20260503_121831',
    '12__g013__20260503_121918',
    '12__g016__20260503_122602',
    '12__g020__20260503_124012',
    '12__g020__20260503_124030',
    '12__g020__20260503_124051',
    '12__g021__20260503_124926',
    '12__g021__20260503_124930',
    '12__g021__20260503_124938',
    '12__g021__20260503_124953',
    '12__g021__20260503_125002',
    '12__g021__20260503_125006',
    '12__g021__20260503_125007',
    '12__g027__20260503_130848',
    '12__g027__20260503_130908',
    '12__g035__20260503_133643',
    '12__g035__20260503_133705',
    '12__g035__20260503_133726',
    '12__g035__20260503_133731',
    '12__g035__20260503_133734',
    '12__g035__20260503_133738',
    '12__g035__20260503_133743',
    '12__g043__20260503_135920',
    '12__g043__20260503_135926',
    '12__g043__20260503_135940',
    '12__g043__20260503_135947',
    '12__g043__20260503_135955',
    '12__g043__20260503_140015',
    '12__g043__20260503_140022',
    '12__g043__20260503_140038',
    '12__g044__20260503_140120',
    '12__g044__20260503_140137',
    '12__g044__20260503_140143',
    '12__g052__20260503_142832',
    '12__g052__20260503_142836',
    '12__g052__20260503_142840',
    '12__g052__20260503_142900',
    '12__g053__20260503_142931',
    '12__g053__20260503_142936',
    '12__g053__20260503_142946',
    '12__g057__20260503_143722',
    '12__g057__20260503_143725',
    '12__g057__20260503_143728',
    '12__g057__20260503_143737',
    '12__g057__20260503_143741',
    '12__g060__20260503_144459',
    '12__g060__20260503_144505',
    '12__g060__20260503_144514',
    '16__g001__20260503_160359',
    '16__g001__20260503_160406',
    '16__g004__20260503_160943',
    '16__g004__20260503_160946',
    '16__g014__20260503_162651',
    '16__g015__20260503_162828',
    '16__g015__20260503_162845',
    '16__g015__20260503_162856',
    '16__g015__20260503_162858',
    '16__g018__20260503_163323',
    '16__g018__20260503_163338',
    '16__g022__20260503_164137',
    '16__g022__20260503_164139',
    '16__g022__20260503_164151',
    '16__g022__20260503_164209',
    '16__g023__20260503_164307',
    '16__g023__20260503_164325',
    '16__g023__20260503_164345',
    '16__g023__20260503_164348',
    '16__g026__20260503_165015',
    '16__g026__20260503_165030',
    '16__g029__20260503_165433',
    '16__g029__20260503_165439',
    '16__g037__20260503_171052',
    '16__g037__20260503_171054',
    '8__g001__20260503_080603',
    '8__g013__20260503_082304',
    '8__g015__20260503_082606',
    '8__g016__20260503_082703',
    '8__g017__20260503_082751',
    '8__g017__20260503_082808',
    '8__g028__20260503_084215',
    '8__g028__20260503_084228',
    '8__g032__20260503_084748',
    '8__g032__20260503_084807',
    '8__g044__20260503_091011_001',
    '8__g045__20260503_091145',
    '8__g049__20260503_091559',
    '8__g049__20260503_091616',
    '8__g055__20260503_092220',
    '8__g058__20260503_092629',
    '8__g058__20260503_092658',
    '8__g062__20260503_093426',
    '8__g063__20260503_093730',
    '8__g067__20260503_094210',
    '8__g070__20260503_094708',
])
print(f'Test fijo: {len(fijos)} imagenes, {sum(nombre_base(p) in fijos for p in SPLITS["test"])} presentes en esta version')

## Entrenar

Todos los modelos usan las mismas opciones y el aumento de datos por defecto de Ultralytics, aplicado
solo a train. Cada corrida (unos 45–60 min con GPU T4) se guarda en Drive.

In [ ]:
import gc
import json
import time

import torch
from ultralytics import YOLO


def carpeta_corrida(modelo, semilla):
    return RES / 'corridas' / f'{Path(modelo).stem}_s{semilla}'


def entrenar(modelo, semilla):
    d = carpeta_corrida(modelo, semilla)
    if (d / 'terminado.json').exists():
        print(f'[listo] {d.name}')
        return
    t0 = time.time()
    ultimo = d / 'weights' / 'last.pt'
    try:
        if ultimo.exists():
            print(f'[continua] {d.name}')
            YOLO(str(ultimo)).train(resume=True)
        else:
            print(f'[entrena] {d.name}')
            YOLO(modelo).train(
                data=str(DATA), epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
                seed=semilla, deterministic=True, device=DEVICE, workers=2,
                project=str(d.parent), name=d.name, exist_ok=True,
            )
    except AssertionError as e:
        # Ultralytics no reanuda una corrida que ya llego al final
        if 'finished' not in str(e) and 'nothing to resume' not in str(e):
            raise
    (d / 'terminado.json').write_text(json.dumps({'modelo': modelo, 'semilla': semilla, 'segundos': time.time() - t0}))
    gc.collect()
    torch.cuda.empty_cache()


for semilla in SEMILLAS:
    for modelo in MODELOS:
        entrenar(modelo, semilla)
print('Entrenamiento completo')

## Evaluar

Por cada corrida terminada mide **validación, test, cada franja horaria y el test fijo**: mAP@0.5,
mAP@0.5:0.95, precisión y recall, total y por clase. También calcula el **acierto de la clase principal**
con los mismos umbrales de la app (conf 0.30, IoU 0.45, NMS agnóstico), que incluye intervalo de confianza
de Wilson.

Si interrumpes el entrenamiento, esta celda evalúa lo que ya terminó.

In [ ]:
import math

import pandas as pd
from PIL import ImageOps

SANA = CLASES.index('sana')

PRUEBAS = {'valid': (DATA, 'val'), 'test': (DATA, 'test')}
for f in ('manana', 'mediodia', 'tarde'):
    rutas = [p for p in SPLITS['test'] if periodo(p) == f]
    if rutas:
        PRUEBAS[f'test_{f}'] = (subconjunto(f'test_{f}', rutas), 'test')
PRUEBAS['test_fijo'] = (subconjunto('test_fijo', [p for p in SPLITS['test'] if nombre_base(p) in fijos]), 'test')

RUTAS_ACIERTO = {
    'test': SPLITS['test'],
    **{f'test_{f}': [p for p in SPLITS['test'] if periodo(p) == f] for f in ('manana', 'mediodia', 'tarde')},
}


def wilson(ok, n, z=1.96):
    if n == 0:
        return (float('nan'), float('nan'))
    p = ok / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    m = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - m, c + m)


def acierto_principal(model):
    # Misma regla que la app: deteccion de mayor confianza; sin detecciones cuenta como sana
    pred = {}
    for p in SPLITS['test']:
        if not clases_en(p):
            continue
        im = ImageOps.exif_transpose(Image.open(p)).convert('RGB')
        b = model(im, conf=0.30, iou=0.45, imgsz=IMGSZ, agnostic_nms=True, verbose=False)[0].boxes
        top = int(b.cls[b.conf.argmax()]) if len(b) else SANA
        pred[p] = top == collections.Counter(clases_en(p)).most_common(1)[0][0]
    filas = []
    for nombre, rutas in RUTAS_ACIERTO.items():
        r = [pred[p] for p in rutas if p in pred]
        lo, hi = wilson(sum(r), len(r))
        filas.append({'prueba': nombre, 'imagenes': len(r), 'acierto': sum(r) / len(r) if r else float('nan'), 'ic95_bajo': lo, 'ic95_alto': hi})
    return filas


general, por_clase, acierto = [], [], []
corridas = sorted(d for d in (RES / 'corridas').iterdir() if (d / 'terminado.json').exists())
assert corridas, 'Todavia no hay corridas terminadas'
for d in corridas:
    arq, sem = d.name.rsplit('_s', 1)
    model = YOLO(str(d / 'weights' / 'best.pt'))
    base = {'modelo': arq, 'semilla': int(sem), 'corrida': d.name}
    for nombre, (data, split) in PRUEBAS.items():
        m = model.val(data=str(data), split=split, imgsz=IMGSZ, batch=BATCH, device=DEVICE, plots=True,
                      project=str(d / 'evaluacion'), name=nombre, exist_ok=True, verbose=False)
        general.append({**base, 'prueba': nombre, 'mAP50': m.box.map50, 'mAP50_95': m.box.map, 'precision': m.box.mp, 'recall': m.box.mr})
        for i, k in enumerate(m.box.ap_class_index):
            por_clase.append({**base, 'prueba': nombre, 'clase': CLASES[int(k)], 'precision': m.box.p[i], 'recall': m.box.r[i],
                              'AP50': m.box.ap50[i], 'AP50_95': m.box.ap[i]})
    acierto += [{**base, **f} for f in acierto_principal(model)]
    del model
    gc.collect()
    torch.cuda.empty_cache()

general, por_clase, acierto = pd.DataFrame(general), pd.DataFrame(por_clase), pd.DataFrame(acierto)
general.to_csv(RES / 'metricas_generales.csv', index=False)
por_clase.to_csv(RES / 'metricas_por_clase.csv', index=False)
acierto.to_csv(RES / 'acierto_clase_principal.csv', index=False)
print('Guardado en', RES)

## Resultados y elección del modelo

El modelo se elige con **validación**, no con test: si se elige con test, esa medida deja de ser honesta.

In [ ]:
def resumen(df, por, cols):
    g = df.groupby(por)[cols]
    r = g.mean().round(4).astype(str) + ' ± ' + g.std().fillna(0).round(4).astype(str)
    r.insert(0, 'corridas', df.groupby(por).size())
    return r


cols = ['mAP50', 'mAP50_95', 'precision', 'recall']
print('Media ± desviación entre semillas\n')
for prueba in PRUEBAS:
    print(f'--- {prueba} ---')
    display(resumen(general[general.prueba == prueba], 'modelo', cols))

print('--- Test por clase ---')
display(resumen(por_clase[por_clase.prueba == 'test'], ['modelo', 'clase'], ['precision', 'recall', 'AP50', 'AP50_95']))

print('--- Acierto de la clase principal (reglas de la app) ---')
display(resumen(acierto, ['modelo', 'prueba'], ['acierto']))

val = general[general.prueba == 'valid']
arq = val.groupby('modelo').mAP50_95.mean().idxmax()
mejor = val[val.modelo == arq].sort_values('mAP50_95', ascending=False).iloc[0]
print(f"\nElegido por validacion: {arq} (corrida {mejor.corrida}, mAP50-95 valid = {mejor.mAP50_95:.4f})")
if arq.startswith('yolov10'):
    print('Aviso: la salida de YOLOv10 es distinta a la de v8/v11; la app necesitara un ajuste para usarlo en el celular.')

## Guardar el paquete de resultados

In [ ]:
import shutil
import zipfile

destino = RES / 'mejor_modelo'
destino.mkdir(exist_ok=True)
shutil.copy2(RES / 'corridas' / mejor.corrida / 'weights' / 'best.pt', destino / 'best.pt')
fila = lambda p: general[(general.corrida == mejor.corrida) & (general.prueba == p)].iloc[0]
(destino / 'datos.json').write_text(json.dumps({
    'nombre': NOMBRE, 'modelo': arq, 'corrida': mejor.corrida, 'version_roboflow': VERSION,
    'imgsz': IMGSZ, 'clases': CLASES,
    'valid_mAP50_95': float(fila('valid').mAP50_95), 'test_mAP50': float(fila('test').mAP50),
    'test_mAP50_95': float(fila('test').mAP50_95),
}, indent=2))

paquete = RES / f'resultados_{NOMBRE}.zip'
with zipfile.ZipFile(paquete, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in RES.rglob('*'):
        if not f.is_file() or f == paquete:
            continue
        # De los pesos solo el mejor modelo; de cada corrida, sus curvas y resultados
        if f.suffix == '.pt' and destino not in f.parents:
            continue
        z.write(f, f.relative_to(RES))
print(f'Listo: {paquete} ({paquete.stat().st_size / 1e6:.1f} MB)')